# Independent Monte Carlo validation: do GLR or BOCPD improve decisions?

Earlier notebooks established that CUSUM, a one-sided Gaussian GLR and a conjugate normal BOCPD can run sequentially. Passing a smoke test is not evidence of superior statistical or economic performance.

This experiment distinguishes independent null calibration, whole-stream false-alarm rates, familywise false alarms when we inspect three detectors, and *paired* economic regret on common simulated streams.

A finite-horizon threshold has a conditional experimental meaning; no conclusion extends automatically to endless monitoring, unknown serial dependence or causal effects.


In [ ]:
import os
from IPython.display import display
import matplotlib.pyplot as plt
from comparative_change_detection import ComparisonDesign, run_comparison
from detector_evidence_audit import (
    empirical_null_alarms, paired_regret,
    run_sensitivity, summarize_sensitivity
)
SMOKE = os.environ.get("MC_VALIDATION_SMOKE", "0") == "1"
REPEATS = int(os.environ.get("MC_VALIDATION_REPEATS", "10"))
if REPEATS < 1:
    raise ValueError("REPEATS must be positive")
design = ComparisonDesign(
 n_fit=80 if SMOKE else 160,n_cal=35 if SMOKE else 80,
 n_live=45 if SMOKE else 120,null_streams=19 if SMOKE else 99,
 delays=(0,) if SMOKE else (0,12),refit_window=70 if SMOKE else 100,
 glr_max_window=16 if SMOKE else 32,
 action_costs=(0.6,) if SMOKE else (0.2,0.6,1.0,1.4),
 refit_costs=(0.1,) if SMOKE else (0.,0.05,0.25)
)
calibration, models, economics = run_comparison(
 design,evaluation_repeats=REPEATS,
 scenarios=("low_variance_signal",) if SMOKE else ("low_variance_signal","high_variance_signal"),
 regimes=("stable","gradual_drift") if SMOKE else ("stable","gradual_drift","ar1","gradual_ar1")
)
display(calibration)


## Independent streams, paired regret

Economic regret includes actual retraining charges. We subtract the CUSUM regret from each candidate's regret *within the same simulated stream and economic cost*. Negative differences favour the candidate. The percentile bootstrap resamples stream-level differences; it never treats adjacent time observations or alternative cost thresholds as independent.


In [ ]:
paired=paired_regret(economics,bootstrap_draws=100 if SMOKE else 600)
display(paired.head(20).round(3))
alarm=empirical_null_alarms(models)
display(alarm.round(3))


## Detector multiplicity

Calibrating CUSUM, GLR and BOCPD at 10% each does not ensure that the probability that *at least one* detector alarms is 10%. We compare individually calibrated thresholds against a Bonferroni threshold targeting 10%/3 per detector.

For exchangeable entire null-stream maxima, this bounds the fixed-horizon three-detector alarm family without requiring detector independence. It does not correct for post-hoc profile selection or preserve a guarantee under a misspecified dependence model.


In [ ]:
sensitivity_design=design
cal_s,obs_s=run_sensitivity(
 sensitivity_design,
 evaluation_repeats=1 if SMOKE else max(20,REPEATS),
 profiles=("Baseline",) if SMOKE else ("Baseline","Short memory","Long memory"),
 regimes=("stable","gradual_drift") if SMOKE else ("stable","gradual_drift","ar1","gradual_ar1"),
 seed=7026,
)
summary=summarize_sensitivity(obs_s)
display(cal_s.head(20).round(3))
display(summary.loc[summary.detector=="Any detector"].round(3))


## Sensitivity and interpretation

Short- and long-memory profiles alter the GLR scan window, CUSUM allowance and BOCPD hazard together; these are deliberately fixed *before* evaluation, so we cannot attribute changes to a single parameter. On gradual-drift streams, first-alarm latency is reported only when the first alarm occurs after the drift begins; streams without a detection are censored, not assigned zero delay.

Default simulations remain exploratory. A publication-grade analysis should considerably increase independent calibration and evaluation runs, include a separate parameter-selection corpus, and show precision around claims of improvement. This work compares predictive maintenance under known synthetic conditional means, not causal treatment benefits.
